In [109]:
!pip install icrawler

In [ ]:
# 이미지 다운로드 하기
from icrawler.builtin import BingImageCrawler
import os

def donwload_img(save_path, keyword, max_num):
	# 저장할 폴더 생성
	# save_path = '../../data/dataset_imagenet/tteokbokki'
	save_path = save_path
	if not os.path.exists(save_path):
		os.makedirs(save_path)

	# 크롤러 설정 (Bing 크롤러가 구글보다 제약이 적어 자주 사용됩니다)
	bing_crawler = BingImageCrawler(storage={'root_dir': save_path})
	bing_crawler.crawl(keyword=keyword, max_num=max_num)

	print(f"다운로드가 완료되었습니다. 저장 경로: {save_path}")

# donwload_img('../../data/dataset_imagenet/kimbap','김밥', 100)
# donwload_img('../../data/dataset_imagenet/tteokbokki','떡볶이', 100)

In [110]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# [1, 2, 3] # 리스트
# {1, 2, 3} # set
# {'name' : 'hong'} # 딕셔너리
# (1, 2) # 튜플

# 하이퍼파라미터 설정
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
DATA_DIR = '../../data/dataset_imagenet'

# 이미지 데이터 생성
# 학습 데이터
train_ds = tf.keras.utils.image_dataset_from_directory(
  DATA_DIR,
  validation_split=0.2,
  subset = 'training',
  seed = 1000,
  image_size = IMG_SIZE,
  batch_size = BATCH_SIZE,
  color_mode='rgb'
)

# 검증 데이터
val_ds = tf.keras.utils.image_dataset_from_directory(
  DATA_DIR,
  validation_split=0.2,
  subset = 'validation',
  seed = 1000,
  image_size = IMG_SIZE,
  batch_size = BATCH_SIZE,
  color_mode='rgb'
)

class_names = train_ds.class_names



Found 59 files belonging to 2 classes.
Using 48 files for training.
Found 59 files belonging to 2 classes.
Using 11 files for validation.


In [ ]:
# 성능 최적화
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(60).prefetch(buffer_size=AUTOTUNE) # 학습할땐 편향 방지를 위해 섞음
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [112]:
# 모델 설계
# 전이학습 모델 가져옴
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top = False, weights = 'imagenet')

# 데이터 증강 -> 이미지를 돌리고 확대하고 움직여서 여러 이미지를 추가하는 작업
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip('horizontal_and_vertical'),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2)
])

# 새 모델 설계
model = models.Sequential([
  layers.Input((224, 224, 3)),
  data_augmentation,
  # 정규화를 0~1이 아닌 -1~1로 해야함. 왜? imagenet에서 그렇기 했기 때문에 맞춰줘야 함
  layers.Rescaling(1./127.5, offset = -1),
  base_model,
  layers.GlobalAveragePooling2D(),
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.2),
  layers.Dense(2, activation='softmax') # 떡볶이, 김밥이라 2 / 나중에 추가할수도있어서 softmax 
])

In [113]:
# 모델설정
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [114]:
callbacks = [
	# 조기 종료
	EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
	# 가장 좋았던 성능을 저장 => 중간에 중지해도 중지전까지 최고 성능을 가져옴
	ModelCheckpoint(filepath='전이학습_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
	# 학습률을 고정 값이 아닌 상황에 따라 줄어들도록 조절
	ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

In [115]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1, callbacks=callbacks)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7292 - loss: 0.5642  
Epoch 1: val_accuracy improved from None to 0.81818, saving model to 전이학습_model.keras

Epoch 1: finished saving model to 전이학습_model.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 41s 6s/step - accuracy: 0.8333 - loss: 0.4306 - val_accuracy: 0.8182 - val_loss: 0.3133 - learning_rate: 0.0010
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9896 - loss: 0.0654
Epoch 2: val_accuracy did not improve from 0.81818
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step - accuracy: 0.9792 - loss: 0.0624 - val_accuracy: 0.8182 - val_loss: 0.2399 - learning_rate: 0.0010
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9583 - loss: 0.0431
Epoch 3: val_accuracy improved from 0.81818 to 1.00000, saving model to 전이학습_model.keras

Epoch 3: finished saving model to 전이학습_model.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 4s/step - accuracy: 0.9792 - loss: 0.0254 - val_accuracy: 1.0000 - val_loss: 0.0677 - learning_rate: 0.0010
Epoch 4/10
2/2 ━

In [116]:
import requests
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import mobilenet_v2
from io import BytesIO


def predicts_url(url):
	# 이미지 정보를 가져옴
	response = requests.get(url)
	# 가져온 이미지를 바이트 형태로 변환
	img = image.load_img(BytesIO(response.content), target_size=(224,224))

	# 이미지 전처리 => 입력 형태 맞추기, 이미지를 배열로
	X = image.img_to_array(img)
	X = np.expand_dims(X, axis=0)

	# 예측
	predicts = model.predict(X)
	
	print('-'*30)
	for i in range(2):
		predict = predicts[0][i]
		print(f'{class_names[i]} : {predict*100:.2f}%')
	print('-'*30)

In [117]:

# 김밥
predicts_url("https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTQ7YEHXzRKi7SdrnNhpt_x75vtRrRiUPaA5A&s")

# 떡볶이
predicts_url("https://mblogthumb-phinf.pstatic.net/MjAyNDAyMDFfMTE4/MDAxNzA2NzYyNDYxMzQ3.AWMYuAVIcsRXADoQ_-EjwHWjfJtNs6UGGJtJMnXjOLog.79Cqs-AYDTAX-cSP-r7o8jSL_1rrl2Y6rB4DcoXF2osg.JPEG.baby0817/BAck_DDUk001koma_ghy112.jpg?type=w800")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
------------------------------
kimbap : 100.00%
tteokbokki : 0.00%
------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
------------------------------
kimbap : 0.41%
tteokbokki : 99.59%
------------------------------
